# Comtrade vs IMF Trade Discrepancy Analysis

**Hypothesis:** There are significant discrepancies between Comtrade and IMF Direction of Trade Statistics (DOTS) for comparable trade variables, which limits the reliability of these sources.

This notebook runs the **same pipeline** as `scripts/run_trade_discrepancy_analysis.py` via `trade_discrepancy.pipeline.run_analysis`, then displays the results interactively.

**Analysis dimensions**
1. Metadata discrepancies (definitions, schema, coverage, partners)
2. Coverage & comparability
3. Data harmonization
4. Discrepancy magnitude
5. Temporal trends
6. Structural partner / flow decomposition

**Data sources**
- UN Comtrade (`data/NEW COMTRADE data/TradeData_sitc4_ag3_2000_2024.csv`) — SITC Rev.4 AG3 commodity × partner × flow, observed 2008–2024 (AUS, CHN, USA, World)
- IMF Pacific DOTS (`data/IMF data/IMF_Pacific_DOTS.csv`) — annual exports/imports by partner, 2000–2024


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "trade_discrepancy").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from trade_discrepancy.constants import OUTPUT_CSV_DIR, OUTPUT_DIR, OUTPUT_PLOTS_DIR
from trade_discrepancy.pipeline import ANALYSIS_DIMENSIONS, run_analysis

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Shared pipeline dimensions:")
for dim in ANALYSIS_DIMENSIONS:
    print(f"  - {dim}")
print(f"CSV dir: {OUTPUT_CSV_DIR}")
print(f"Plots dir: {OUTPUT_PLOTS_DIR}")


## 1. Run full analysis pipeline

Calls `run_analysis()` — identical entry point to the CLI script. Writes CSVs to `outputs/trade_discrepancy/csv/` and plots to `outputs/trade_discrepancy/plots/`.


In [ ]:
results = run_analysis(OUTPUT_DIR)

metrics = results["metrics"]
summary = results["summary"]
by_year = results["by_year"]
coverage = results["coverage"]
availability = results["availability"]
by_partner = results["by_partner"]
top_world_gaps = results["top_world_gaps"]
metadata_coverage = results["metadata_coverage"]
valuation_completeness = results["valuation_completeness"]
classification_grain = results["classification_grain"]
reporter_coverage = results["reporter_coverage"]

print(f"Comparable observations: {results['n_comparable_observations']:,}")
print(f"World observations: {results['n_world_observations']:,}")
print(f"Median world SymDiff%: {results['median_world_symmetric_pct_diff']:.2f}%")
print(f"Mean abs SymDiff% (world): {results['mean_abs_world_symmetric_pct_diff']:.2f}%")
print(
    f"Share within {results['tolerance_pct']:.0f}% (world): "
    f"{results['share_world_within_tolerance']:.1%}"
)
print(f"CSV dir: {results['csv_dir']}")
print(f"Plots dir: {results['plots_dir']}")
print(f"Outputs written to: {results['output_dir']}")


## 2. Coverage and metadata tables

Year-by-year record counts in the Premium extract, overlap with IMF, then metadata tables (valuation completeness, commodity grain, reporter sets). Simple comparisons stay as tables.


In [ ]:
display(availability)
display(coverage)
display(metadata_coverage)
display(valuation_completeness)
display(classification_grain)
display(reporter_coverage)


## 3. Headline metrics by partner

World totals vs bilateral Australia / China.


In [ ]:
display(by_partner)

world = metrics[metrics["partner"] == "world"].copy()
world_pivot = world.pivot_table(
    index=["country", "year"],
    columns="flow",
    values="symmetric_pct_diff",
)
world_pivot.sort_values(["country", "year"]).head(20)


## 4. Discrepancy summary by country, flow, and partner


In [ ]:
summary.sort_values("median_symmetric_pct_diff", key=abs, ascending=False)


## 5. Temporal trends (world totals)


In [ ]:
by_year_world = by_year[by_year["partner"] == "world"]
by_year_world


## 6. Largest world-total discrepancies


In [ ]:
display(top_world_gaps)


## 7. Core visualizations

Plots that tables cannot replace: world-total scatter, median-discrepancy heatmap, layered value overlays, and SymDiff% time series.


In [ ]:
for path in results["plots"]:
    path = Path(path)
    print(path.name)
    display(Image(filename=str(path)))


## 8. Output inventory

CSVs are under outputs/trade_discrepancy/csv/; plots under outputs/trade_discrepancy/plots/.


In [ ]:
for label, folder in [("CSV", OUTPUT_CSV_DIR), ("Plots", OUTPUT_PLOTS_DIR)]:
    files = sorted(folder.glob("*"))
    print(f"{label}: {len(files)} files in {folder}")
    for path in files:
        print(f"  {path.name}")


## 9. Interpretation notes

**Core patterns**
1. **Bilateral AUS/CHN/USA often agree closely** — when Comtrade has matching partner data, those series are frequently within ±5%.
2. **World totals diverge more often** — only about half of world-total comparisons fall within ±5%.
3. **Gaps are country-specific** — Fiji tracks closely; Palau 2017 world imports, later Tonga world exports, and Solomon Islands China imports do not.
4. **Reliability is not universal** — neither source is automatically better; assess by country, flow, and partner.

Re-run via CLI: python scripts/run_trade_discrepancy_analysis.py
